# TEM-Style 4x4 Training on Colab

This notebook trains the 4x4 next-state task using the original TEM-style sequential loop: observation remapping, path integration, Hebbian memory, and a supervised next-state head.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

for package in ["scipy", "tensorboard"]:
    ensure_package(package)

print("Dependencies ready.")

In [ ]:
import os

repo_dir = "/content/torch_tem"
if not os.path.exists(repo_dir):
    !git clone https://github.com/YifeiCAO/torch_tem.git {repo_dir}
else:
    print(f"Repository already exists at {repo_dir}")
    !git -C {repo_dir} pull

%cd /content/torch_tem

In [ ]:
import importlib
import matplotlib.pyplot as plt
import torch

import extract_tem_representations
import train_2d_tem_style
importlib.reload(extract_tem_representations)
importlib.reload(train_2d_tem_style)

collect_representations = extract_tem_representations.collect_representations
train = train_2d_tem_style.train
load_model = train_2d_tem_style.load_model

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
TRAIN_STEPS = 2000
BATCH_SIZE = 16
ROLLOUT_LENGTH = 20
SUPERVISED_WEIGHT = 1.0
TEM_WEIGHT = 0.2
REMAP_STRATEGY = "fixed"  # try "curriculum" after this is stable
REMAP_CURRICULUM_STEPS = 1000
WALK_LENGTH_MIN_CHUNKS = 5
WALK_LENGTH_MAX_CHUNKS = 15
REPRESENTATION_CHUNKS = 100
SEED = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "/content/torch_tem/tem_2d_style.pt"
REPRESENTATION_PATH = "/content/torch_tem/tem_2d_representations.pt"

In [ ]:
predictor = train(
    train_steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE,
    rollout_length=ROLLOUT_LENGTH,
    supervised_weight=SUPERVISED_WEIGHT,
    tem_weight=TEM_WEIGHT,
    remap_strategy=REMAP_STRATEGY,
    remap_curriculum_steps=REMAP_CURRICULUM_STEPS,
    walk_length_min_chunks=WALK_LENGTH_MIN_CHUNKS,
    walk_length_max_chunks=WALK_LENGTH_MAX_CHUNKS,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every=250,
    seed=SEED,
    device=DEVICE,
)

print(f"Saved checkpoint to {CHECKPOINT_PATH}")

In [ ]:
loaded_predictor, train_config, _ = load_model(
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

print("Loaded checkpoint.")
print(train_config)

In [ ]:
representations = collect_representations(
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=BATCH_SIZE,
    rollout_length=ROLLOUT_LENGTH,
    num_chunks=REPRESENTATION_CHUNKS,
    remap_strategy="fixed",
    walk_length_min_chunks=WALK_LENGTH_MIN_CHUNKS,
    walk_length_max_chunks=WALK_LENGTH_MAX_CHUNKS,
    seed=SEED,
    device=DEVICE,
)

torch.save(representations, REPRESENTATION_PATH)
g_vectors = representations["g_vectors"]
p_vectors = representations["p_vectors"]
counts = representations["counts"]

print(f"Saved representations to {REPRESENTATION_PATH}")
print("g_vectors shape:", tuple(g_vectors.shape))
print("p_vectors shape:", tuple(p_vectors.shape))

In [ ]:
print("Visits per node:")
print(counts)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

im0 = axes[0].imshow(g_vectors.numpy(), aspect="auto", cmap="viridis")
axes[0].set_title("Mean g vectors across 16 grid nodes")
axes[0].set_ylabel("node index")
axes[0].set_xlabel("g dimension")
fig.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.02)

im1 = axes[1].imshow(p_vectors.numpy(), aspect="auto", cmap="magma")
axes[1].set_title("Mean p vectors across 16 grid nodes")
axes[1].set_ylabel("node index")
axes[1].set_xlabel("p dimension")
fig.colorbar(im1, ax=axes[1], fraction=0.02, pad=0.02)

plt.show()

In [ ]:
def project_to_2d(vectors):
    centered = vectors - vectors.mean(dim=0, keepdim=True)
    U, S, V = torch.pca_lowrank(centered, q=2)
    return centered @ V[:, :2]

g_2d = project_to_2d(g_vectors)
p_2d = project_to_2d(p_vectors)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

axes[0].scatter(g_2d[:, 0].numpy(), g_2d[:, 1].numpy(), s=60)
for i in range(16):
    axes[0].text(g_2d[i, 0].item(), g_2d[i, 1].item(), str(i))
axes[0].set_title("PCA of mean g vectors")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(p_2d[:, 0].numpy(), p_2d[:, 1].numpy(), s=60)
for i in range(16):
    axes[1].text(p_2d[i, 0].item(), p_2d[i, 1].item(), str(i))
axes[1].set_title("PCA of mean p vectors")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.show()